In [1]:
import os
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import v2
from torchvision import tv_tensors
from torchvision.tv_tensors import BoundingBoxes
from torchvision.models import ResNet50_Weights
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchmetrics.detection.mean_ap import MeanAveragePrecision

In [2]:
import os
import random
import numpy as np
import torch


SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


generator = torch.Generator()
generator.manual_seed(SEED)

In [3]:
# =========================================================
# Config
# =========================================================

ROOT = Path(
    "/kaggle/input/competitions/payt-olp-ai-training-image-detection"
)

IMG_TRAIN = ROOT / "data/train/images"
LAB_TRAIN = ROOT / "data/train/labels"
IMG_VAL = ROOT / "data/val/images"
LAB_VAL = ROOT / "data/val/labels"
IMG_TEST = ROOT / "data/test/images"
SUB_PATH = ROOT / "sample_submission.csv"

BATCH_SIZE = 4
EPOCHS = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:", DEVICE)

DEVICE: cuda


In [4]:
# =========================================================
# YOLO normalized -> Faster R-CNN xyxy
# =========================================================

def yolo_to_frcnn(text, width, height):
    boxes, labels = [], []

    for line in text.splitlines():
        parts = line.split()

        if len(parts) < 5:
            continue

        cls = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:5])

        xc, bw = xc * width, bw * width
        yc, bh = yc * height, bh * height

        x1 = max(0.0, min(xc - bw / 2, width))
        y1 = max(0.0, min(yc - bh / 2, height))
        x2 = max(0.0, min(xc + bw / 2, width))
        y2 = max(0.0, min(yc + bh / 2, height))

        if x2 > x1 and y2 > y1:
            boxes.append([x1, y1, x2, y2])
            labels.append(cls + 1)  # 0 dành cho background

    return boxes, labels

In [5]:
# =========================================================
# Ghép ảnh và label theo stem
# =========================================================

def load_samples(img_dir, label_dir):
    image_map = {
        file.stem: file.name
        for file in img_dir.iterdir()
        if file.suffix.lower() in {".jpg", ".jpeg", ".png"}
    }

    samples = []

    for label_file in sorted(label_dir.glob("*.txt")):
        image_name = image_map.get(label_file.stem)

        if image_name is None:
            continue

        image_path = img_dir / image_name
        with Image.open(image_path) as img:
            width, height = img.size

        boxes, labels = yolo_to_frcnn(label_file.read_text(), width=width, height=height)

        if boxes:
            samples.append((image_name, boxes, labels))

    return samples


train_samples = load_samples(IMG_TRAIN, LAB_TRAIN)
val_samples = load_samples(IMG_VAL, LAB_VAL)



sample_submission = pd.read_csv(SUB_PATH, dtype={"image_id": str})

test_file_map = {
    file.stem: file.name
    for file in IMG_TEST.iterdir()
    if file.suffix.lower() in {".jpg", ".jpeg", ".png"}
}

missing_ids = [
    image_id
    for image_id in sample_submission["image_id"]
    if image_id not in test_file_map
]

if missing_ids:
    raise FileNotFoundError(f"Không tìm thấy ảnh test: {missing_ids[:5]}")

test_files = [
    test_file_map[image_id]
    for image_id in sample_submission["image_id"]
]

print("Train:", len(train_samples))
print("Val:", len(val_samples))
print("Test:", len(test_files))

Train: 424
Val: 68
Test: 164


In [6]:
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import tv_tensors


class DetectionDataset(Dataset):
    def __init__(self, samples, img_dir, transform, is_test=False):
        self.samples = samples
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if self.is_test:
            image_name = self.samples[idx]
            image = Image.open(self.img_dir / image_name).convert("RGB")
            if self.transform is not None:
                image = self.transform(image)
            return image, image_name

        image_name, boxes, labels = self.samples[idx]

        # Đọc ảnh
        image = Image.open(self.img_dir / image_name).convert("RGB")
        w, h = image.size

        # Định dạng boxes và labels
        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(labels, dtype=torch.int64).reshape(-1)

        boxes = tv_tensors.BoundingBoxes(
            boxes,
            format=tv_tensors.BoundingBoxFormat.XYXY,
            canvas_size=(h, w),
        )

        # Gom target thành dict để SanitizeBoundingBoxes có thể đồng bộ boxes và labels
        target = {
            "boxes": boxes,
            "labels": labels,
        }

        if self.transform is not None:
            image, target = self.transform(image, target)

        # Chuyển boxes về Tensor chuẩn cho mô hình detection
        boxes = torch.as_tensor(target["boxes"], dtype=torch.float32).reshape(-1, 4)
        labels = target["labels"]

        # Tính toán area và metadata sau khi đã qua biến đổi hình học / lọc box
        if len(boxes) > 0:
            area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        else:
            area = torch.zeros((0,), dtype=torch.float32)

        final_target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx], dtype=torch.int64),
            "area": area,
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
        }

        return image, final_target

def collate_fn(batch):
    return tuple(zip(*batch))


train_transform = v2.Compose([
    v2.ToImage(),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomAffine(degrees=5,translate=(0.05, 0.05),scale=(0.95, 1.05),),
    v2.ColorJitter(brightness=0.15,contrast=0.15,saturation=0.15,hue=0.03,),
    v2.ToDtype(torch.float32, scale=True),
    v2.ClampBoundingBoxes(),
    v2.SanitizeBoundingBoxes(min_size=1.0),
])


val_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

train_data = DetectionDataset(
    train_samples, IMG_TRAIN, train_transform
)

val_data = DetectionDataset(
    val_samples, IMG_VAL, val_transform
)

test_data = DetectionDataset(
    test_files, IMG_TEST, val_transform, is_test=True
)

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=True,
    num_workers=4
)

val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=True,
    num_workers=4
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=True,
)

In [7]:
def evaluate_loss_and_map(model, data_loader, metric, device):
    val_loss = 0.0
    metric.reset()

    with torch.no_grad():
        # ================= LOSS =================
        model.train()

        for images, targets in data_loader:
            images = [image.to(device) for image in images]
            targets_device = [
                {k: v.to(device) for k, v in t.items()}
                for t in targets
            ]

            loss_dict = model(images, targets_device)
            val_loss += sum(loss_dict.values()).item()

        val_loss /= len(data_loader)

        # ================= mAP =================
        model.eval()

        for images, targets in data_loader:
            images = [image.to(device) for image in images]

            predictions = model(images)

            formatted_preds = [
                {k: v.cpu() for k, v in pred.items()}
                for pred in predictions
            ]

            formatted_targets = [
                {
                    "boxes": t["boxes"].cpu(),
                    "labels": t["labels"].cpu(),
                }
                for t in targets
            ]

            metric.update(
                formatted_preds,
                formatted_targets,
            )

    summary = metric.compute()

    return (
        val_loss,
        summary["map"].item(),
        summary["map_50"].item(),
        summary["map_75"].item(),
    )

In [8]:
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
)

from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT

model = fasterrcnn_resnet50_fpn_v2(
    weights=weights,
    min_size=1024,
    max_size=1536,
)

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    num_classes=2
)

model = model.to(DEVICE)

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth


100%|██████████| 167M/167M [00:00<00:00, 200MB/s] 


In [9]:
# =========================================================
# STAGE 1 - FREEZE BACKBONE
# =========================================================

print("=" * 80)
print("STAGE 1: TRAIN DETECTION HEADS")
print("=" * 80)

# Freeze backbone + FPN
for p in model.backbone.parameters():
    p.requires_grad = False


# Chỉ đưa parameter trainable vào optimizer
trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer = AdamW(
    trainable_params,
    lr=1e-4,
    weight_decay=1e-4,
)


scheduler = ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    threshold=1e-4,
)


metric = MeanAveragePrecision(
    box_format="xyxy",
    class_metrics=False,
)


STAGE1_EPOCHS = 5

best_map_stage1 = -1.0


for epoch in range(STAGE1_EPOCHS):

    # =====================================================
    # TRAIN
    # =====================================================

    model.train()

    train_loss = 0.0

    for images, targets in tqdm(
        train_loader,
        desc=f"Stage 1 [{epoch + 1:02d}/{STAGE1_EPOCHS}]"
    ):

        images = [
            image.to(DEVICE)
            for image in images
        ]

        targets = [
            {
                k: v.to(DEVICE)
                for k, v in t.items()
            }
            for t in targets
        ]


        optimizer.zero_grad(set_to_none=True)

        loss_dict = model(
            images,
            targets,
        )

        loss = sum(
            loss_dict.values()
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()


    train_loss /= len(train_loader)


    # =====================================================
    # VALIDATION
    # =====================================================

    (
        val_loss,
        val_map,
        val_map_50,
        val_map_75,
    ) = evaluate_loss_and_map(
        model,
        val_loader,
        metric,
        DEVICE,
    )


    scheduler.step(val_map)

    current_lr = optimizer.param_groups[0]["lr"]


    # =====================================================
    # LOGGING
    # =====================================================

    print("\n" + "=" * 80)

    print(
        f"STAGE 1 | "
        f"EPOCH [{epoch + 1:02d}/{STAGE1_EPOCHS}] "
        f"| LR: {current_lr:.2e}"
    )

    print("-" * 80)

    print(
        f"Train Loss : {train_loss:.4f}"
    )

    print(
        f"Val Loss   : {val_loss:.4f}"
    )

    print(
        f"Val mAP    : {val_map:.4f}"
    )

    print(
        f"Val mAP@50 : {val_map_50:.4f}"
    )

    print(
        f"Val mAP@75 : {val_map_75:.4f}"
    )

    print("-" * 80)


    # =====================================================
    # SAVE BEST MODEL
    # =====================================================

    if val_map > best_map_stage1:

        best_map_stage1 = val_map

        torch.save(
            model.state_dict(),
            "best_model_stage1.pth",
        )

        print(
            f"[SAVED] Best Stage 1 model "
            f"(mAP={val_map:.4f})"
        )

    print("=" * 80 + "\n")

STAGE 1: TRAIN DETECTION HEADS


Stage 1 [01/5]: 100%|██████████| 106/106 [02:07<00:00,  1.20s/it]



STAGE 1 | EPOCH [01/5] | LR: 1.00e-04
--------------------------------------------------------------------------------
Train Loss : 0.0978
Val Loss   : 0.0483
Val mAP    : 0.7930
Val mAP@50 : 0.9541
Val mAP@75 : 0.9352
--------------------------------------------------------------------------------
[SAVED] Best Stage 1 model (mAP=0.7930)



Stage 1 [02/5]: 100%|██████████| 106/106 [02:13<00:00,  1.26s/it]



STAGE 1 | EPOCH [02/5] | LR: 1.00e-04
--------------------------------------------------------------------------------
Train Loss : 0.0453
Val Loss   : 0.0512
Val mAP    : 0.7698
Val mAP@50 : 0.9717
Val mAP@75 : 0.9717
--------------------------------------------------------------------------------



...


STAGE 1 | EPOCH [03/5] | LR: 1.00e-04
--------------------------------------------------------------------------------
Train Loss : 0.0407
Val Loss   : 0.0418
Val mAP    : 0.8133
Val mAP@50 : 0.9767
Val mAP@75 : 0.9767
--------------------------------------------------------------------------------
[SAVED] Best Stage 1 model (mAP=0.8133)



Stage 1 [04/5]: 100%|██████████| 106/106 [02:13<00:00,  1.26s/it]



STAGE 1 | EPOCH [04/5] | LR: 1.00e-04
--------------------------------------------------------------------------------
Train Loss : 0.0376
Val Loss   : 0.0390
Val mAP    : 0.8172
Val mAP@50 : 0.9842
Val mAP@75 : 0.9657
--------------------------------------------------------------------------------
[SAVED] Best Stage 1 model (mAP=0.8172)



Stage 1 [05/5]: 100%|██████████| 106/106 [02:13<00:00,  1.26s/it]



STAGE 1 | EPOCH [05/5] | LR: 1.00e-04
--------------------------------------------------------------------------------
Train Loss : 0.0359
Val Loss   : 0.0420
Val mAP    : 0.8191
Val mAP@50 : 0.9891
Val mAP@75 : 0.9891
--------------------------------------------------------------------------------
[SAVED] Best Stage 1 model (mAP=0.8191)



In [10]:
import shutil

shutil.copy(
    "best_model_stage1.pth",
    "best_model.pth"
)

'best_model.pth'

In [11]:
# =========================================================
# STAGE 2 - FULL FINE-TUNING
# =========================================================

print("=" * 80)
print("STAGE 2: FULL MODEL FINE-TUNING")
print("=" * 80)


# =========================================================
# Load best Stage 1
# =========================================================

model.load_state_dict(
    torch.load(
        "best_model_stage1.pth",
        map_location=DEVICE,
    )
)


# =========================================================
# UNFREEZE BACKBONE
# =========================================================

for p in model.backbone.parameters():
    p.requires_grad = True


# =========================================================
# Separate backbone / detection parameters
# =========================================================

backbone_params = list(
    model.backbone.parameters()
)

backbone_param_ids = {
    id(p)
    for p in backbone_params
}


head_params = [
    p
    for p in model.parameters()
    if id(p) not in backbone_param_ids
]


# =========================================================
# Different LR
# =========================================================

optimizer = AdamW(
    [
        {
            "params": backbone_params,
            "lr": 1e-5,
        },
        {
            "params": head_params,
            "lr": 5e-5,
        },
    ],
    weight_decay=1e-4,
)


scheduler = ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    threshold=1e-4,
)


metric = MeanAveragePrecision(
    box_format="xyxy",
    class_metrics=False,
)


# Full fine-tuning cần lâu hơn Stage 1
STAGE2_EPOCHS = 20

EARLY_STOPPING_PATIENCE = 6

patient_counter = 0

best_map_stage2 = best_map_stage1


# =========================================================
# TRAIN
# =========================================================

for epoch in range(STAGE2_EPOCHS):

    model.train()

    train_loss = 0.0


    for images, targets in tqdm(
        train_loader,
        desc=f"Stage 2 [{epoch + 1:02d}/{STAGE2_EPOCHS}]"
    ):

        images = [
            image.to(DEVICE)
            for image in images
        ]

        targets = [
            {
                k: v.to(DEVICE)
                for k, v in t.items()
            }
            for t in targets
        ]


        optimizer.zero_grad(
            set_to_none=True
        )


        loss_dict = model(
            images,
            targets,
        )


        loss = sum(
            loss_dict.values()
        )


        loss.backward()


        # Optional nhưng khá hữu ích
        # khi fine-tune detector dataset nhỏ
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0,
        )


        optimizer.step()


        train_loss += loss.item()


    train_loss /= len(train_loader)


    # =====================================================
    # VALIDATION
    # =====================================================

    (
        val_loss,
        val_map,
        val_map_50,
        val_map_75,
    ) = evaluate_loss_and_map(
        model,
        val_loader,
        metric,
        DEVICE,
    )


    scheduler.step(val_map)


    backbone_lr = optimizer.param_groups[0]["lr"]
    head_lr = optimizer.param_groups[1]["lr"]


    # =====================================================
    # LOGGING
    # =====================================================

    print("\n" + "=" * 80)

    print(
        f"STAGE 2 | "
        f"EPOCH [{epoch + 1:02d}/{STAGE2_EPOCHS}]"
    )

    print(
        f"Backbone LR : {backbone_lr:.2e}"
    )

    print(
        f"Head LR     : {head_lr:.2e}"
    )

    print("-" * 80)

    print(
        f"Train Loss : {train_loss:.4f}"
    )

    print(
        f"Val Loss   : {val_loss:.4f}"
    )

    print(
        f"Val mAP    : {val_map:.4f}"
    )

    print(
        f"Val mAP@50 : {val_map_50:.4f}"
    )

    print(
        f"Val mAP@75 : {val_map_75:.4f}"
    )

    print("-" * 80)


    # =====================================================
    # CHECKPOINT
    # =====================================================

    if val_map > best_map_stage2:

        best_map_stage2 = val_map

        patient_counter = 0

        torch.save(
            model.state_dict(),
            "best_model.pth",
        )

        print(
            f"[SAVED] New best model "
            f"(mAP={val_map:.4f})"
        )

    else:

        patient_counter += 1

        print(
            f"[NO IMPROVEMENT] "
            f"{patient_counter}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )


        if (
            patient_counter
            >= EARLY_STOPPING_PATIENCE
        ):

            print(
                "\n[STOP] Early stopping"
            )

            break


    print("=" * 80 + "\n")

STAGE 2: FULL MODEL FINE-TUNING


Stage 2 [01/20]: 100%|██████████| 106/106 [05:53<00:00,  3.34s/it]



STAGE 2 | EPOCH [01/20]
Backbone LR : 1.00e-05
Head LR     : 5.00e-05
--------------------------------------------------------------------------------
Train Loss : 0.0336
Val Loss   : 0.0331
Val mAP    : 0.8509
Val mAP@50 : 0.9958
Val mAP@75 : 0.9958
--------------------------------------------------------------------------------
[SAVED] New best model (mAP=0.8509)



Stage 2 [02/20]: 100%|██████████| 106/106 [05:53<00:00,  3.33s/it]



STAGE 2 | EPOCH [02/20]
Backbone LR : 1.00e-05
Head LR     : 5.00e-05
--------------------------------------------------------------------------------
Train Loss : 0.0287
Val Loss   : 0.0341
Val mAP    : 0.8440
Val mAP@50 : 0.9865
Val mAP@75 : 0.9865
--------------------------------------------------------------------------------
[NO IMPROVEMENT] 1/6



...


STAGE 2 | EPOCH [08/20]
Backbone LR : 1.00e-05
Head LR     : 5.00e-05
--------------------------------------------------------------------------------
Train Loss : 0.0235
Val Loss   : 0.0283
Val mAP    : 0.8634
Val mAP@50 : 0.9982
Val mAP@75 : 0.9982
--------------------------------------------------------------------------------
[NO IMPROVEMENT] 2/6



Stage 2 [09/20]: 100%|██████████| 106/106 [05:53<00:00,  3.33s/it]



STAGE 2 | EPOCH [09/20]
Backbone LR : 5.00e-06
Head LR     : 2.50e-05
--------------------------------------------------------------------------------
Train Loss : 0.0231
Val Loss   : 0.0459
Val mAP    : 0.7838
Val mAP@50 : 0.9910
Val mAP@75 : 0.9910
--------------------------------------------------------------------------------
[NO IMPROVEMENT] 3/6



Stage 2 [10/20]:   6%|▌         | 6/106 [00:26<07:29,  4.50s/it]


KeyboardInterrupt: 

In [12]:
# =========================================================
# Inference
# =========================================================

model.load_state_dict(
    torch.load("best_model.pth", map_location=DEVICE)
)
model.eval()

submission_rows = []

with torch.no_grad():
    for images, image_names in tqdm(
        test_loader,
        desc="Predicting",
    ):
        images = [image.to(DEVICE) for image in images]
        predictions = model(images)

        for prediction, image, image_name in zip(
            predictions,
            images,
            image_names,
        ):
            height, width = image.shape[-2:]

            boxes = prediction["boxes"].cpu()
            scores = prediction["scores"].cpu()

            keep = scores > 0
            boxes = boxes[keep]
            scores = scores[keep]

            if len(boxes) == 0:
                prediction_string = (
                    "0 0.500000 0.500000 "
                    "0.500000 0.100000 0.100000"
                )
            else:
                predictions_text = []

                for box, score in zip(boxes, scores):
                    x1, y1, x2, y2 = box.tolist()

                    xc = ((x1 + x2) / 2) / width
                    yc = ((y1 + y2) / 2) / height
                    bw = (x2 - x1) / width
                    bh = (y2 - y1) / height

                    predictions_text.append(
                        f"0 {score.item():.6f} "
                        f"{xc:.6f} {yc:.6f} "
                        f"{bw:.6f} {bh:.6f}"
                    )

                prediction_string = " ".join(predictions_text)

            submission_rows.append({
                "image_id": Path(image_name).stem,
                "prediction_string": prediction_string,
            })


submission = pd.DataFrame(submission_rows)
submission.to_csv("submission.csv", index=False)

print("Saved submission.csv")

Predicting: 100%|██████████| 41/41 [00:58<00:00,  1.42s/it]

Saved submission.csv
